# Step 7 (rebuilt) — Joint Per-Class Policy Search

Redone against D-071/D-072. The prior version searched UNIFORM line-wide grid points and cherry-picked, per SKU, whichever run happened to be cheapest for it — the combination it reported was never actually simulated, and capacity was never verified for it. **That result is retracted.**

This notebook instead searches **per-class combinations directly**, and every candidate is **one real joint simulation**. Whatever comes out as "optimal" is a plan that was actually, jointly run — infeasible candidates are excluded before cost is even compared, not discovered after the fact.

**Runtime:** ~50s per line at n=3, ~3.5 minutes for all four lines. Grid is 3^6 = 729 combinations per line.

**Scope, stated plainly:** `forecast_bias_correction` and `min_run_hours` are held at fixed scalars during this search — not an engine limitation (D-071 made both per-class-capable), but because searching all four levers per class is 3^12 combinations, infeasible in the time available. Cover and service are what drives the capacity-feasibility question this redo exists to answer.

## Setup

In [ ]:
import subprocess, os, sys
def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0: print("STDERR:", r.stderr[-1500:])
    return r
REPO = '/content/ibp-tradeoff'
os.chdir('/content'); sh('rm -rf ibp-tradeoff')
sh('git clone https://github.com/rdelolmog-creator/ibp-tradeoff.git')
os.chdir(REPO); sys.path.insert(0, REPO)
print('cwd:', os.getcwd())

## Rebuild the Step 4 artefacts

In [ ]:
import pandas as pd, numpy as np, yaml
from src.ingest import DataIngestor
from src.cleaner import DataCleaner
if not os.path.isdir('data_primary/raw'):
    sh('python generate_data.py'); sh('mv data data_primary')
ing = DataIngestor(repo_root=REPO, data_root='data_primary')
clean_master, sku_master, _ = DataCleaner(ing.schema, ing.assumptions).clean(ing.load())
os.makedirs('data_primary/clean', exist_ok=True)
clean_master.to_parquet('data_primary/clean/clean_master.parquet', index=False)
sku_master.to_parquet('data_primary/clean/sku_master.parquet', index=False)
print('clean_master:', clean_master.shape, '| sku_master:', sku_master.shape)

## Upload Step 5a's output

In [ ]:
from google.colab import files
import shutil
print('Select demand_characteristics.csv AND censoring_diagnostics.csv:')
uploaded = files.upload()
demand_characteristics = pd.read_csv('demand_characteristics.csv').set_index('sku_id')
censoring_diagnostics  = pd.read_csv('censoring_diagnostics.csv').set_index('sku_id')
from src.portfolio_impact import get_flagged_skus
flagged = get_flagged_skus(censoring_diagnostics.reset_index())
for f in ('demand_characteristics.csv', 'censoring_diagnostics.csv'):
    shutil.copy(f, f'data_primary/clean/{f}')
print(f'demand_characteristics: {demand_characteristics.shape} | flagged: {len(flagged)}')

## Build the engine

In [ ]:
from src.engine import TradeOffEngine, LeverSettings, build_line_master
from src.policy_model import (search_all_lines, expand_to_sku_level,
                              sku_level_for_policy_model, PolicyModel,
                              ATTRIBUTE_FEATURES, HISTORY_DERIVED_FEATURES, TARGETS)
assumptions = yaml.safe_load(open('config/assumptions.yaml'))
schema      = yaml.safe_load(open('config/schema.yaml'))
engine = TradeOffEngine(assumptions, schema, clean_master, sku_master,
                        demand_characteristics, flagged)
line_master = build_line_master(assumptions, schema)
print('assumption fingerprint:', engine.assumption_fingerprint)

## The joint search — every candidate a real simulation

`best_feasible_policy` filters to `capacity_shortfall_total <= 1e-6` FIRST, then picks the minimum cost among the feasible rows. The true per-class default (each class at its own default, not class A applied to everyone — D-072's "Base") is guaranteed to be an evaluated candidate, so the optimum can never come out worse than it.

In [ ]:
import time
t0 = time.time()
line_results = search_all_lines(engine, n=3)
print(f'search_all_lines(n=3) took {time.time()-t0:.0f}s')
line_results.to_csv('line_results.csv', index=False)

cols = ['line_id','cover_A','cover_B','cover_C','service_A','service_B','service_C',
        'total_economic_cost_eur','default_total_cost_eur','saving_eur','capacity_shortfall_total']
print(line_results[cols].round(2).to_string(index=False))
print(f"\ntotal saving vs true per-class default: EUR {line_results.saving_eur.sum():,.0f}")
print(f"minimum saving (must be >= 0): {line_results.saving_eur.min():,.2f}")
print(f"any infeasible optimum returned (must be 0): {(line_results.capacity_shortfall_total > 1e-6).sum()}")

## Expand to SKU level — each SKU inherits its own class's policy

In [ ]:
sku_level = expand_to_sku_level(line_results, engine)
sku_level.to_csv('sku_level.csv', index=False)
print(f'{len(sku_level)} SKUs (expect 60)')
print(sku_level.groupby(['line_id','abc_class']).size())

conv_check = sku_level.groupby('line_id')['line_conversion_cost_eur'].nunique()
print(f'\ndistinct conversion values per line (expect all 1 — proves it is NOT split per SKU):')
print(conv_check)

## The ML model — attribute-fitted policy

`sku_level_for_policy_model` reshapes the new joint-search output into what `PolicyModel` expects. `PolicyModel` itself is unchanged by D-071 — only the column names feeding it changed.

**Expect, and this is the honest result, not a weak one:** with policy now set by class, `optimal_cover_weeks`'s naive baseline (predict the ABC-class mean) may score EXACTLY zero error — if the search happened to find the same combination optimal on every line, cover is literally constant within a class across the whole portfolio, and no model can "beat" a baseline that is already perfect by construction. If you see `naive_mae: 0.0`, that is what happened — check `line_results` above to confirm.

In [ ]:
compat = sku_level_for_policy_model(sku_level)
model = PolicyModel(compat, sku_master, demand_characteristics, line_master)
print('training rows:', len(model.training_frame()))
print('FULL features :', len(model.feature_columns('FULL')))
print('LAUNCH        :', len(model.feature_columns('LAUNCH')),
      '(excludes', HISTORY_DERIVED_FEATURES, ')')

evaluation = model.evaluate()
evaluation.to_csv('step07_model_comparison.csv', index=False)
print()
print(evaluation.round(4).to_string(index=False))

## SHAP — why a SKU wants the policy it wants

In [ ]:
import shap
model.fit_final()
for target in TARGETS:
    sv, X = model.explain(target, feature_set='LAUNCH')
    imp = (pd.DataFrame({'feature': X.columns, 'mean_abs_shap': np.abs(sv).mean(axis=0)})
           .sort_values('mean_abs_shap', ascending=False))
    print(f'{target}:')
    print(imp.head(8).round(4).to_string(index=False))
    print()

## The operational output — a new launch

In [ ]:
new_sku = {
    'category': 'personal_care', 'abc_class': 'B', 'gross_margin_eur': 2.10,
    'price_eur': 4.20, 'std_cost_eur': 2.10, 'shelf_life_days': 1092,
    'case_size': 12, 'moq_units': 5000, 'min_run_units': 31500,
    'line_speed_units_hr': 3500,
}
rec = model.predict_launch(new_sku)
print('RECOMMENDED OPENING POLICY')
for k, v in rec.items(): print(f'  {k:<26}{v:>10.3f}')
try:
    model.predict_launch({'category': 'personal_care'})
    print('WARNING: incomplete attributes did NOT raise')
except Exception as e:
    print(f'incomplete attributes correctly refused -> {type(e).__name__}')

## Tests

In [ ]:
sh('python -m pytest tests/test_policy_model.py -q --no-header')
sh('python -m pytest tests/test_engine.py tests/test_pipeline.py -q --no-header')

## Save outputs to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, shutil
out_dir = '/content/drive/My Drive/ibp-tradeoff-outputs'
os.makedirs(out_dir, exist_ok=True)
for f in ['line_results.csv', 'sku_level.csv', 'step07_model_comparison.csv']:
    if os.path.exists(f): shutil.copy(f, out_dir)
print('copied to', out_dir)

## Consolidated report — the only cell to copy

In [ ]:
import hashlib, subprocess

t_pol = subprocess.run('python -m pytest tests/test_policy_model.py -q --no-header',
                       shell=True, capture_output=True, text=True)
t_eng = subprocess.run('python -m pytest tests/test_engine.py -q --no-header',
                       shell=True, capture_output=True, text=True)
if not os.path.isdir('data_control/raw'):
    subprocess.run('cp config/assumptions.yaml config/_bk.yaml && '
                   'cp config/assumptions_lowcensoring.yaml config/assumptions.yaml && '
                   'python generate_data.py && '
                   'cp config/_bk.yaml config/assumptions.yaml && rm config/_bk.yaml && '
                   'mv data data_control', shell=True, capture_output=True, text=True)
t_pipe = subprocess.run('python -m pytest tests/test_pipeline.py -q --no-header',
                        shell=True, capture_output=True, text=True)

L=[]; w=L.append
w('='*78); w('STEP 7 (REBUILT) - JOINT PER-CLASS SEARCH - CONSOLIDATED REPORT'); w('='*78)
w(f'assumption set    : {engine.assumption_fingerprint}')
w(f'engine.py sha     : {hashlib.sha256(open("src/engine.py","rb").read()).hexdigest()[:12]}')
w(f'policy_model sha  : {hashlib.sha256(open("src/policy_model.py","rb").read()).hexdigest()[:12]}')
w(f'pandas {pd.__version__} / numpy {np.__version__}')

w(''); w('-- 1. LINE RESULTS - the feasible optimum per line '+'-'*24)
w(line_results[cols].round(2).to_string(index=False))
w(f'\\ntotal saving vs true per-class default: EUR {line_results.saving_eur.sum():,.0f}')

w(''); w('-- 2. FEASIBILITY CHECK '+'-'*54)
w(f'minimum saving_eur (must be >= 0)              : {line_results.saving_eur.min():,.2f}')
w(f'rows with capacity_shortfall_total > 0 (must be 0): {(line_results.capacity_shortfall_total > 1e-6).sum()}')

w(''); w('-- 3. SKU EXPANSION '+'-'*57)
w(f'sku_level rows: {len(sku_level)} (expect 60)')
w(str(sku_level.groupby(["line_id","abc_class"]).size()))
w(f'\\ndistinct conversion values per line (expect all 1): {dict(conv_check)}')

w(''); w('-- 4. MODEL COMPARISON '+'-'*55)
w(evaluation.round(4).to_string(index=False))
w('')
w('naive_mae near/at 0.0 for a target = that target is (near) constant')
w('within class across the portfolio at this grid resolution - the naive')
w('ABC-class-mean baseline cannot be beaten because it is already exact.')
w('This is expected under D-071s design, not a modelling failure.')

w(''); w('-- 5. LAUNCH RECOMMENDATION '+'-'*49)
for k, v in rec.items(): w(f'  {k:<28}{v:>10.3f}')

w(''); w('-- 6. TESTS '+'-'*66)
for label, r in (('test_policy_model.py', t_pol), ('test_engine.py', t_eng),
                 ('test_pipeline.py', t_pipe)):
    w(f'{label:<24}: ' + (r.stdout.strip().splitlines() or ["no output"])[-1])
if any(r.returncode for r in (t_pol, t_eng, t_pipe)):
    w(''); w('FAILURES:')
    for r in (t_pol, t_eng, t_pipe):
        if r.returncode: w(r.stdout[-2500:])

w(''); w('-- 7. CHECKS '+'-'*65)
checks = [
 ('every optimum is capacity-feasible', bool((line_results.capacity_shortfall_total <= 1e-6).all())),
 ('optimum never worse than the true per-class default', bool(line_results.saving_eur.min() >= -1e-6)),
 ('conversion cost is a line total, not split per SKU', bool(conv_check.eq(1).all())),
 ('all test suites pass', all(r.returncode == 0 for r in (t_pol, t_eng, t_pipe))),
]
for label, ok in checks:
    w(f'  [{"PASS" if ok else "SEE NOTE"}]  {label}')
w('')
w('Magnitudes are not findings (arch section 10). Every number above is')
w('whatever the generator and the assumption set encoded.')
w('='*78)

report_text = '\n'.join(L)
open('step07_report.txt','w').write(report_text)
try:
    import shutil
    d = '/content/drive/My Drive/ibp-tradeoff-outputs'
    if os.path.isdir(d):
        for f in ('step07_report.txt','line_results.csv','sku_level.csv',
                  'step07_model_comparison.csv'):
            if os.path.exists(f): shutil.copy(f, d)
        print('saved to', d, '\\n')
except Exception as e:
    print('Drive copy skipped:', e, '\\n')
print(report_text)